# Validação MLflow + S3

Notebook de validação da integração MLflow Tracking Server com LocalStack S3.

**Pré-requisitos:** `docker compose up -d` a correr.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("forest-risk-fire-prediction")
print("MLflow URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name("forest-risk-fire-prediction"))

In [ ]:
X, y = make_classification(n_samples=500, n_features=8, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="baseline-rf"):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    f1 = f1_score(y_test, model.predict(X_test))

    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("f1_score", f1)
    mlflow.sklearn.log_model(model, "model")
    print(f"F1: {f1:.3f}")
    print(f"Run registado no MLflow: http://localhost:5000")

In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://localstack:4566",
    aws_access_key_id="test",
    aws_secret_access_key="test",
    region_name="eu-west-1",
)

response = s3.list_objects_v2(Bucket="forest-risk-models", Prefix="mlflow/")
for obj in response.get("Contents", []):
    print(obj["Key"])

## Resultado esperado

- Experimento `forest-risk-fire-prediction` visível em http://localhost:5000
- Artefactos do modelo em `s3://forest-risk-models/mlflow/<experiment_id>/<run_id>/artifacts/model/`
- F1 score > 0.80 (dados sintéticos com make_classification)